# ToonVerse — ConvNeXt V2-Large (v3) · multietiqueta + crossovers sintéticos

Clasificador **multietiqueta** de universos de caricaturas (detecta crossovers) sobre **imagen subida**.

**v3 — enseña crossovers SIN dataset de crossovers reales.**
El modelo ya era multietiqueta (sigmoid + `BCEWithLogitsLoss`), pero en entrenamiento **solo veía imágenes de UNA serie a la vez** → en una imagen con dos universos (p.ej. Simpson + Star Wars) encendía solo uno y disparaba falsos positivos (Naruto/One Piece sin estar). La causa: nunca vio un ejemplo con dos clases presentes a la vez. v3 fabrica los crossovers **al vuelo** desde los datasets independientes que ya tienes:
- **Mosaico 2×2** (`MOSAIC_PROB`): pega 4 series distintas en cuadrantes; etiqueta = **unión** de las 4. Imita el caso "póster con personajes de varias series" — la señal de crossover más fuerte.
- **MixUp** (`MIXUP_PROB`): mezcla 2 series por transparencia; etiqueta = **unión**. Refuerza la co-ocurrencia en el espacio de etiquetas.
- **`otra` se mantiene como clase de fondo**: en una imagen combinada solo vale `1` si NINGUNA serie está presente (no se ensucia al mezclar).
- En inferencia se decide con **umbral por clase** → pueden encender varias a la vez.
- Las imágenes individuales conservan su etiqueta única → la detección no-crossover sigue intacta.
- **Estabilidad (heredado del 04):** gradient clipping (norma `GRAD_CLIP=1.0`) + `RandomErasing` en la augmentación.

> La validación es single-label, así que el F1/mAP de val **no** mide crossovers. Para verlos, está la **Celda 10b** (sanity-check sobre mosaicos sintéticos).

---
**v2 (previo) arregló 3 cosas de la corrida v1 (F1 0.74, backbone sin entrenar):**
1. **Bug de `torch.compile`** — en v1 el backbone NO se entrenó (cambiar `requires_grad` a media corrida no recompilaba el backward; el loss quedó clavado en 1.2222). Ahora se fija la config de entrenables **antes** de compilar y se entrena en **una sola fase**.
2. **pos_weight 25 → 6** — la descalibración daba F1@0.5 = 0.23 (umbrales caricaturas ~0.88, otra ~0.45).
3. **`otra` 800k → 300k** + augmentación más suave (en caricaturas el color importa).

**Ajustado para L4 (24 GB):** afina **solo el último stage** (`UNFREEZE_FROM_STAGE=3`) para que entrene de verdad sin reventar tu ventana de ~12 h (full fine-tuning de Large en L4 son ~2 h/época). Checkpoint en Drive cada época: re-ejecuta la celda de entrenamiento para reanudar.

> Guarda como `toonverse_convnextv2L_v3.pt` — **no toca** tus modelos anteriores (v1/v2).

## Celda 1 — Setup + dependencias

In [ ]:
# ── Dependencias de performance + timm (ConvNeXt V2) ──────────
!pip install -q timm scikit-learn kornia PyTurboJPEG
!apt-get install -y libturbojpeg0-dev -q 2>/dev/null | tail -1

import os
# Menos fragmentación de VRAM → menos OOM con modelos grandes
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import io, time, shutil, zipfile, warnings, threading
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from PIL import Image
from sklearn.metrics import f1_score, average_precision_score
from scipy.special import expit
import kornia.augmentation as K
import matplotlib.pyplot as plt
import timm
from google.colab import drive

warnings.filterwarnings('ignore')
drive.mount('/content/drive', force_remount=False)

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_BF16    = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_COMPILE = torch.__version__ >= '2.0.0'

torch.set_float32_matmul_precision('high')   # TF32 en A100
torch.backends.cudnn.benchmark = True

print(f"PyTorch : {torch.__version__} | timm : {timm.__version__}")
print(f"Device  : {DEVICE}")
if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    ram  = os.popen("free -g | awk '/^Mem/{print $2}'").read().strip()
    print(f"GPU     : {gpu} ({vram:.0f} GB) | RAM ~{ram} GB")
    print(f"BF16    : {'OK' if USE_BF16 else 'NO'} | Compile: {'OK' if USE_COMPILE else 'NO'}")
    if 'A100' not in gpu:
        print("AVISO: no es A100 — los tiempos serán mayores")
else:
    print("SIN GPU -> Entorno de ejecución > Cambiar tipo de entorno > A100")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 132.0 MB/s eta 0:00:00

Mounted at /content/drive
PyTorch : 2.11.0+cu128 | timm : 1.0.27
Device  : cuda
GPU     : NVIDIA A100-SXM4-80GB (85 GB) | RAM ~167 GB
BF16    : OK | Compile: OK


## Celda 2 — Configuración (rutas, modelo, hiperparámetros)

In [ ]:
# ════ VERSIÓN ════
VERSION = '3.0'  # ConvNeXt V2-Large + crossovers sintéticos (mosaico+mixup) + estabilidad

# ════ RUTAS (corregidas: Mi unidad / ExpoEscom / zip) ════
ZIP_ROOT      = '/content/drive/MyDrive/ExpoEscom/zip'
LOCAL_DATASET = '/content/dataset_local'
SAVE_DIR      = '/content/drive/MyDrive/ExpoEscom/models'
MODEL_FILE    = 'toonverse_convnextv2L_v3.pt'              # <- v3: NO pisa v1/v2

os.makedirs(SAVE_DIR, exist_ok=True)
SAVE_PATH   = os.path.join(SAVE_DIR, MODEL_FILE)               # mejor modelo (para la app)
CKPT_RESUME = os.path.join(SAVE_DIR, 'toonverse_resume_v3.pt') # estado para reanudar (v3)

# Nombre de archivo en Drive -> carpeta de clase
ZIP_MAP = {
    'pokemon.zip':'pokemon', 'one_piece.zip':'one_piece', 'dragon_ball.zip':'dragon_ball',
    'bleach.zip':'bleach', 'ben_10.zip':'ben_10', 'hora_de_aventura.zip':'hora_de_aventura',
    'unshowmas.zip':'un_show_mas', 'naruto.zip':'naruto', 'doraemon.zip':'doraemon',
    'yugioh.zip':'yu_gi_oh', 'barbie.zip':'barbie', 'aot.zip':'attack_on_titan',
    'simpson.zip':'simpson', 'star_wars.zip':'star_wars',
}
OTRA_ZIP     = 'otra.zip'
# otra SUBMUESTREADO a 300k (antes 800k): rebalancea (deja de ser el 50% del dataset) y acelera
OTRA_SUBCATS = {'cartoons_anime':120_000, 'real_life':100_000, 'noise':80_000}

# ════ DATASET ════
MAX_PER_CLASS  = 57_143
MAX_OTRA_TOTAL = 300_000         # <- v2 (era 800k)
VAL_SPLIT      = 0.2
SEED           = 42
IMG_SIZE       = 224
IMAGENET_MEAN  = [0.485, 0.456, 0.406]
IMAGENET_STD   = [0.229, 0.224, 0.225]
THRESHOLD      = 0.5

# ════ MODELO ════
MODEL_ARCH = 'convnextv2_large.fcmae_ft_in22k_in1k'
DROP_RATE  = 0.3
DROP_PATH  = 0.1

# ════ HIPERPARÁMETROS — v2 (una sola fase; arregla el bug de compile) ════
# Estás en L4 (24GB, ~3x mas lento que A100). Full fine-tuning de Large NO cabe en tu tiempo.
#   UNFREEZE_FROM_STAGE = 3  -> afina SOLO el ultimo stage (~31% del modelo).
#       El forward sigue siendo completo (manda el tiempo), el backward solo por arriba:
#       costo por epoca parecido al de v1 PERO ahora el modelo si aprende.
#   Si consigues A100, baja a 2 para fine-tuning casi-completo.
BATCH_SIZE  = 96
GRAD_ACCUM  = 3                  # batch efectivo = 288
NUM_WORKERS = min(12, os.cpu_count() or 8)   # ~nucleos reales (no mas que CPUs)
PREFETCH    = 6

NUM_EPOCHS  = 14                 # una sola fase de fine-tuning + early stopping
UNFREEZE_FROM_STAGE = 3          # afina stage 3 + cabeza (stages 0..3). OOM? sube a... no aplica; baja BATCH_SIZE a 64

LR_FT     = 1.2e-4               # con warmup (antes 8e-5 ni movia el backbone)
WD_FT     = 0.05
GRAD_CLIP = 1.0                  # <- v3: clip de norma de gradiente (estabiliza el fine-tuning; heredado del 04)
EARLY_STOP_PATIENCE = 4
LABEL_SMOOTH        = 0.05
POS_WEIGHT_MAX      = 6.0        # <- v2 (era 25: causaba la descalibracion otra=0.45 vs caric=0.88)

# ════ CROSSOVER SINTÉTICO — v3 (fabrica multietiqueta desde datasets independientes) ════
# Cada batch se arma como: mosaico 2x2 (4 series), MixUp (2 series), o limpio (1 serie).
# La etiqueta de un batch combinado es la UNION de las clases presentes -> el modelo
# aprende a encender VARIAS clases a la vez = detectar crossovers.
MOSAIC_PROB = 0.35     # fraccion de batches armados como mosaico 2x2
MIXUP_PROB  = 0.15     # fraccion de batches con MixUp        (mosaico+mixup = 50% sinteticos)
MIXUP_ALPHA = 0.2      # parametro Beta del MixUp
# -> ~50% de batches quedan single-label (no-crossover sigue afilado)

print("Config v3.0 cargada")
print(f"  Arch    : {MODEL_ARCH}  (afina desde stage {UNFREEZE_FROM_STAGE})")
print(f"  Batch   : {BATCH_SIZE} x{GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM} efectivo | workers {NUM_WORKERS}")
print(f"  Epocas  : {NUM_EPOCHS} (una fase) | LR {LR_FT} | clip {GRAD_CLIP} | pos_weight cap {POS_WEIGHT_MAX}")
print(f"  Crossover: mosaico {MOSAIC_PROB:.0%} + mixup {MIXUP_PROB:.0%} (etiqueta union)")
print(f"  otra    : {MAX_OTRA_TOTAL:,} (submuestreado)")
print(f"  Salida  : {SAVE_PATH}")

## Celda 3 — Verificar ZIPs en Drive

In [ ]:
print("="*65); print("VERIFICANDO ZIPs en", ZIP_ROOT)
CLASSES = []
for zf_name, cls in ZIP_MAP.items():
    p = os.path.join(ZIP_ROOT, zf_name)
    if os.path.exists(p):
        print(f"  ok  {cls:22s} <- {zf_name:24s} ({os.path.getsize(p)/1e9:.2f} GB)")
        CLASSES.append(cls)
    else:
        print(f"  XX  {cls:22s} <- {zf_name} NO ENCONTRADO")

otra_path = os.path.join(ZIP_ROOT, OTRA_ZIP)
if os.path.exists(otra_path):
    print(f"  ok  {'otra':22s} <- {OTRA_ZIP:24s} ({os.path.getsize(otra_path)/1e9:.2f} GB)")
    CLASSES.append('otra')
else:
    print(f"  !!  otra <- {OTRA_ZIP} NO ENCONTRADO (entrenaría sin clase 'otra')")

NUM_CLASSES = len(CLASSES)
print("\n" + "="*65)
print(f"Clases ({NUM_CLASSES}): {CLASSES}")
assert NUM_CLASSES >= 2, "No se encontraron ZIPs — revisa ZIP_ROOT en la Celda 2"
if 'A100' not in (torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''):
    print("Recuerda: estos tiempos asumen A100")

VERIFICANDO ZIPs en /content/drive/MyDrive/ExpoEscom/zip
  ok  pokemon                <- pokemon.zip              (0.92 GB)
  ok  one_piece              <- one_piece.zip            (0.94 GB)
  ok  dragon_ball            <- dragon_ball.zip          (2.96 GB)
  ok  bleach                 <- bleach.zip               (1.43 GB)
  ok  ben_10                 <- ben_10.zip               (1.63 GB)
  ok  hora_de_aventura       <- hora_de_aventura.zip     (3.38 GB)
  ok  un_show_mas            <- unshowmas.zip            (1.50 GB)
  ok  naruto                 <- naruto.zip               (1.48 GB)
  ok  doraemon               <- doraemon.zip             (0.77 GB)
  ok  yu_gi_oh               <- yugioh.zip               (1.36 GB)
  ok  barbie                 <- barbie.zip               (1.49 GB)
  ok  attack_on_titan        <- aot.zip                  (1.37 GB)
  ok  simpson                <- simpson.zip              (1.07 GB)
  ok  star_wars              <- star_wars.zip            (0.91 GB)
  ok 

## Celda 4 — Extracción de ZIPs a /content (SSD local)

In [ ]:
# Estrategia: copiar ZIP a /tmp (rápido) y extraer al SSD local de Colab.
# Evita el I/O lento de Drive FUSE durante el entrenamiento.
EXT = ('.jpg', '.jpeg', '.png', '.webp')

def count_local(path):
    n = 0
    for _, _, files in os.walk(path):
        n += sum(1 for f in files if f.lower().endswith(EXT))
    return n

def extract_zip(zip_filename, dst_name, flatten=True):
    src = os.path.join(ZIP_ROOT, zip_filename)
    dst = os.path.join(LOCAL_DATASET, dst_name)
    tmp = f'/tmp/{zip_filename}'
    if not os.path.exists(src):
        print(f"  XX {zip_filename} no encontrado"); return 0
    if os.path.exists(dst) and count_local(dst) > 500:     # reanudar entre sesiones
        n = count_local(dst); print(f"  ok {dst_name}: ya existe ({n:,})"); return n
    os.makedirs(dst, exist_ok=True)
    t0 = time.time()
    print(f"  copiando {zip_filename} ({os.path.getsize(src)/1e9:.2f} GB)...", end=' ', flush=True)
    shutil.copy2(src, tmp)
    print(f"{time.time()-t0:.0f}s | extrayendo...", end=' ', flush=True)
    t1 = time.time()
    with zipfile.ZipFile(tmp, 'r') as zf:
        zf.extractall(dst)
    os.remove(tmp)
    print(f"{time.time()-t1:.0f}s")
    # Aplana si el ZIP trae UNA sola carpeta envolvente (no aplica a otra)
    if flatten:
        top = [c for c in os.listdir(dst) if not c.startswith('.')]
        if len(top) == 1 and os.path.isdir(os.path.join(dst, top[0])):
            sub = os.path.join(dst, top[0])
            for it in os.listdir(sub):
                shutil.move(os.path.join(sub, it), dst)
            os.rmdir(sub)
    return count_local(dst)

os.makedirs(LOCAL_DATASET, exist_ok=True)
t_all = time.time()
print("="*65 + "\nEXTRAYENDO 14 UNIVERSOS\n" + "="*65)
for zf_name, cls in ZIP_MAP.items():
    if cls not in CLASSES:
        continue
    print(f"     -> {cls}: {extract_zip(zf_name, cls, flatten=True):,} imgs")

if 'otra' in CLASSES:
    print("\notra (zip único, conserva subcarpetas):")
    print(f"     -> otra: {extract_zip(OTRA_ZIP, 'otra', flatten=False):,} imgs")

print(f"\nListo en {(time.time()-t_all)/60:.1f} min")
print(os.popen("du -sh /content/dataset_local").read().strip())

EXTRAYENDO 14 UNIVERSOS
  copiando pokemon.zip (0.92 GB)... 27s | extrayendo... 11s
     -> pokemon: 65,534 imgs
  copiando one_piece.zip (0.94 GB)... 21s | extrayendo... 7s
     -> one_piece: 53,571 imgs
  copiando dragon_ball.zip (2.96 GB)... 87s | extrayendo... 31s
     -> dragon_ball: 140,748 imgs
  copiando bleach.zip (1.43 GB)... 42s | extrayendo... 18s
     -> bleach: 100,000 imgs
  copiando ben_10.zip (1.63 GB)... 47s | extrayendo... 19s
     -> ben_10: 100,000 imgs
  copiando hora_de_aventura.zip (3.38 GB)... 103s | extrayendo... 33s
     -> hora_de_aventura: 100,000 imgs
  copiando unshowmas.zip (1.50 GB)... 45s | extrayendo... 18s
     -> un_show_mas: 100,000 imgs
  copiando naruto.zip (1.48 GB)... 46s | extrayendo... 14s
     -> naruto: 65,534 imgs
  copiando doraemon.zip (0.77 GB)... 26s | extrayendo... 11s
     -> doraemon: 65,535 imgs
  copiando yugioh.zip (1.36 GB)... 30s | extrayendo... 8s
     -> yu_gi_oh: 65,535 imgs
  copiando barbie.zip (1.49 GB)... 45s | extrayend

## Celda 5 — Construir muestras + split estratificado

In [ ]:
def build_samples(root_dir, classes, max_per_class, max_otra, otra_subcats, seed=SEED):
    all_samples = []
    rng = np.random.default_rng(seed)
    for cls_idx, cls_name in enumerate(classes):
        cls_dir = os.path.join(root_dir, cls_name)
        if not os.path.exists(cls_dir):
            continue
        label = np.zeros(len(classes), dtype=np.float32)
        label[cls_idx] = 1.0
        if cls_name == 'otra':
            subcats = [s for s in otra_subcats if os.path.isdir(os.path.join(cls_dir, s))]
            imgs = []
            if subcats:   # subcarpetas presentes -> cap por subcategoría
                for sub in subcats:
                    sub_imgs = [os.path.join(cls_dir, sub, f)
                                for f in os.listdir(os.path.join(cls_dir, sub))
                                if f.lower().endswith(EXT)]
                    rng.shuffle(sub_imgs); imgs.extend(sub_imgs[:otra_subcats[sub]])
                rng.shuffle(imgs)
            else:         # sin subcarpetas -> recorrer todo y cap global
                for r, _, files in os.walk(cls_dir):
                    for f in files:
                        if f.lower().endswith(EXT):
                            imgs.append(os.path.join(r, f))
                rng.shuffle(imgs); imgs = imgs[:max_otra]
            print(f"  ok {'otra':22s}: {len(imgs):,} imgs")
        else:
            imgs = [os.path.join(cls_dir, f) for f in os.listdir(cls_dir)
                    if f.lower().endswith(EXT)]
            rng.shuffle(imgs); imgs = imgs[:max_per_class]
            print(f"  ok {cls_name:22s}: {len(imgs):,} imgs")
        for path in imgs:
            all_samples.append((path, label.copy()))
    return all_samples

def stratified_split(all_samples, num_classes, val_ratio=VAL_SPLIT, seed=SEED):
    rng = np.random.default_rng(seed); train, val = [], []
    for c in range(num_classes):
        cls_s = [(p, l) for p, l in all_samples if l[c] == 1.0]
        if not cls_s:
            continue
        rng.shuffle(cls_s); n_val = max(1, int(len(cls_s) * val_ratio))
        val.extend(cls_s[:n_val]); train.extend(cls_s[n_val:])
    rng.shuffle(train); rng.shuffle(val)
    return train, val

print("Construyendo dataset...")
all_samples = build_samples(LOCAL_DATASET, CLASSES, MAX_PER_CLASS, MAX_OTRA_TOTAL, OTRA_SUBCATS)
train_samples, val_samples = stratified_split(all_samples, NUM_CLASSES)
print(f"\nTotal: {len(all_samples):,} | Train: {len(train_samples):,} | Val: {len(val_samples):,}")

Construyendo dataset...
  ok pokemon               : 57,143 imgs
  ok one_piece             : 53,571 imgs
  ok dragon_ball           : 57,143 imgs
  ok bleach                : 57,143 imgs
  ok ben_10                : 57,143 imgs
  ok hora_de_aventura      : 57,143 imgs
  ok un_show_mas           : 57,143 imgs
  ok naruto                : 57,143 imgs
  ok doraemon              : 57,143 imgs
  ok yu_gi_oh              : 57,143 imgs
  ok barbie                : 57,143 imgs
  ok attack_on_titan       : 57,143 imgs
  ok simpson               : 57,143 imgs
  ok star_wars             : 57,143 imgs
  ok otra                  : 300,000 imgs

Total: 1,096,430 | Train: 877,152 | Val: 219,278


## Celda 6 — Caché en RAM + DataLoaders  *(arreglado el bug de `__getitem__`)*

In [ ]:
# TurboJPEG por worker (no es pickleable: se crea dentro de cada worker)
_tl = threading.local()
def _worker_init(wid):
    try:
        from turbojpeg import TurboJPEG
        _tl.jpeg = TurboJPEG()
    except Exception:
        pass  # cae a Pillow si falla

class RAMCachedDataset(Dataset):
    """Carga todos los JPEG como bytes en RAM -> cero I/O de disco al entrenar.
    A ~1.6M imgs 224px (~20KB c/u) son ~30-35 GB: caben de sobra en 167 GB."""
    def __init__(self, samples, transform=None):
        self.transform = transform
        print(f"  Pre-cargando {len(samples):,} imágenes en RAM...")
        t0, self.cache, tot = time.time(), [], 0
        for i, (path, label) in enumerate(samples):
            with open(path, 'rb') as f:
                b = f.read()
            self.cache.append((b, label)); tot += len(b)
            if (i + 1) % 100_000 == 0:
                print(f"    {i+1:,}/{len(samples):,} ({tot/1e9:.1f} GB)", end='\r')
        dt = time.time() - t0
        print(f"  OK {tot/1e9:.1f} GB en RAM ({dt:.0f}s, {tot/max(dt,1)/1e9:.1f} GB/s)")

    def __len__(self):
        return len(self.cache)

    def __getitem__(self, idx):                    # <- FIX: ahora SI es método de la clase
        b, label = self.cache[idx]
        img = None
        if hasattr(_tl, 'jpeg'):
            try:
                arr = _tl.jpeg.decode(b)
                img = Image.fromarray(arr[:, :, ::-1].copy())   # BGR -> RGB
            except Exception:
                img = None
        if img is None:
            try:
                img = Image.open(io.BytesIO(b)).convert('RGB')
            except Exception:
                img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (128, 128, 128))
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.float32)

# Transforms en CPU mínimos (augmentación + normalización van en GPU con Kornia)
transform_train_cpu = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),                 # [0,1] — SIN normalize (lo hace Kornia)
])
transform_val_cpu = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print("Cargando TRAIN en RAM..."); train_ds = RAMCachedDataset(train_samples, transform_train_cpu)
print("Cargando VAL en RAM...");   val_ds   = RAMCachedDataset(val_samples,   transform_val_cpu)

# Val usa el MISMO batch que train para no forzar recompilaciones de torch.compile
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True,
    prefetch_factor=PREFETCH, worker_init_fn=_worker_init, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True,
    prefetch_factor=PREFETCH, worker_init_fn=_worker_init)

print(f"\nDataLoaders | train {len(train_ds):,} ({len(train_loader)} batches) | val {len(val_ds):,}")
print("Nota: en L4, afinando solo stage 3, ~25-35 min/época -> 14 épocas ~6-8 h.")

Cargando TRAIN en RAM...
  Pre-cargando 877,152 imágenes en RAM...
  OK 14.9 GB en RAM (174s, 0.1 GB/s)
Cargando VAL en RAM...
  Pre-cargando 219,278 imágenes en RAM...
  OK 3.7 GB en RAM (35s, 0.1 GB/s)

DataLoaders | train 877,152 (9137 batches) | val 219,278
Nota: en L4, afinando solo stage 3, ~25-35 min/época -> 14 épocas ~6-8 h.


## Celda 7 — Modelo ConvNeXt V2-Large + congelado por fase + compile

In [ ]:
# ConvNeXt V2-Large preentrenado (IN-22k -> IN-1k). timm reemplaza el clasificador.
model = timm.create_model(
    MODEL_ARCH, pretrained=True, num_classes=NUM_CLASSES,
    drop_rate=DROP_RATE, drop_path_rate=DROP_PATH,
)
cfg = timm.data.resolve_model_data_config(model)
print("data_config:", {k: cfg[k] for k in ('input_size', 'mean', 'std')})

def _orig(m):
    """Modulo real aunque este envuelto por torch.compile."""
    return getattr(m, '_orig_mod', m)

def set_trainable(model, unfreeze_from_stage=UNFREEZE_FROM_STAGE):
    """Entrena 'head' + stages >= unfreeze_from_stage; el resto congelado.
    (ConvNeXt usa LayerNorm/GRN, no BatchNorm -> no hace falta el truco de BN.eval)."""
    m = _orig(model)
    for name, p in m.named_parameters():
        train = name.startswith('head')
        if name.startswith('stages.') and int(name.split('.')[1]) >= unfreeze_from_stage:
            train = True
        p.requires_grad_(train)
    n   = sum(p.numel() for p in m.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in m.parameters())
    print(f"   Entrenables: {n:,}/{tot:,} ({100*n/tot:.1f}%)  [stage>={unfreeze_from_stage} + head]")

def trainable_params(model):
    return [p for p in _orig(model).parameters() if p.requires_grad]

model = model.to(DEVICE).to(memory_format=torch.channels_last)

# *** FIX del bug v1 ***: fijar la config de entrenables ANTES de compilar, para que
# torch.compile trace el backward con esos gradientes desde el primer paso.
set_trainable(model)

if USE_COMPILE:
    try:
        model = torch.compile(model)
        print("torch.compile activado (modo default)")
    except Exception as e:
        print(f"compile no disponible ({e}) — sigo en eager")
print(f"Modelo {MODEL_ARCH} | {NUM_CLASSES} clases | channels_last")

model.safetensors:   0%|          | 0.00/792M [00:00<?, ?B/s]

data_config: {'input_size': (3, 224, 224), 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225)}
   Entrenables: 61,670,415/196,442,895 (31.4%)  [stage>=3 + head]
torch.compile activado (modo default)
Modelo convnextv2_large.fcmae_ft_in22k_in1k | 15 clases | channels_last


## Celda 8 — Augmentación GPU, pérdida, métricas, train/eval, checkpoints

In [ ]:
import torch.nn.functional as F

# ── Augmentación + normalización en GPU (Kornia, batch completo) ──
_mean = torch.tensor(IMAGENET_MEAN, device=DEVICE).view(1, 3, 1, 1)
_std  = torch.tensor(IMAGENET_STD,  device=DEVICE).view(1, 3, 1, 1)

# v2: augmentacion mas suave (en caricaturas el color es senal fuerte; antes train_loss >> val_loss)
# v3: + RandomErasing (oclusiones, heredado del 04) -> robustez ante personajes tapados
gpu_aug_train = nn.Sequential(
    K.RandomHorizontalFlip(p=0.5),
    K.RandomResizedCrop((IMG_SIZE, IMG_SIZE), scale=(0.8, 1.0), ratio=(0.85, 1.2)),
    K.RandomRotation(degrees=10),
    K.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.02, p=0.5),
    K.RandomGaussianBlur(kernel_size=(3, 3), sigma=(0.1, 1.0), p=0.2),
    K.RandomErasing(scale=(0.02, 0.2), ratio=(0.3, 3.3), value=0.0, p=0.25),
).to(DEVICE)

def normalize_gpu(x):
    return (x - _mean) / _std

# ════ CROSSOVER SINTÉTICO (v3) — fabrica multietiqueta desde imágenes single-label ════
# Opera sobre el batch ya aumentado y en rango [0,1] (ANTES de normalizar). La etiqueta
# de salida es la UNION de las clases combinadas -> el modelo aprende co-ocurrencia.
OTRA_IDX = CLASSES.index('otra') if 'otra' in CLASSES else -1

def _resolve_otra(lab):
    """'otra' = fondo. En una imagen combinada solo vale 1 si NINGUNA serie esta presente,
    para no ensuciar la clase de fondo al mezclar (otra+pokemon -> pokemon, no ambas)."""
    if OTRA_IDX < 0:
        return lab
    franq = lab.sum(dim=1) - lab[:, OTRA_IDX]           # suma de clases NO-otra
    lab[:, OTRA_IDX] = torch.where(franq > 0,
                                   torch.zeros_like(lab[:, OTRA_IDX]),
                                   lab[:, OTRA_IDX])
    return lab

def _mosaic(imgs, labels):
    """Mosaico 2x2: cada cuadrante es una imagen distinta del batch (downscale a la mitad).
    Imita un poster con personajes de varias series. Etiqueta = union de los 4 cuadrantes."""
    B, C, H, W = imgs.shape
    h, w = H // 2, W // 2
    def q():
        idx = torch.randint(0, B, (B,), device=imgs.device)
        return F.interpolate(imgs[idx], size=(h, w), mode='bilinear', align_corners=False), labels[idx]
    a, la = q(); b, lb = q(); c, lc = q(); d, ld = q()
    top = torch.cat([a, b], dim=3)                      # mitad superior (ancho)
    bot = torch.cat([c, d], dim=3)                      # mitad inferior
    mos = torch.cat([top, bot], dim=2)                  # apila (alto) -> [B,C,H,W]
    lab = torch.maximum(torch.maximum(la, lb), torch.maximum(lc, ld))
    return mos, _resolve_otra(lab)

def _mixup(imgs, labels, alpha=MIXUP_ALPHA):
    """MixUp: mezcla 2 imagenes por transparencia. Etiqueta = union (ambas presentes)."""
    lam  = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(imgs.size(0), device=imgs.device)
    mixed = lam * imgs + (1 - lam) * imgs[perm]
    lab   = torch.maximum(labels, labels[perm])
    return mixed, _resolve_otra(lab)

def mix_crossover(imgs, labels):
    """Con MOSAIC_PROB arma mosaico, con MIXUP_PROB arma mixup, si no deja el batch limpio."""
    r = np.random.rand()
    if r < MOSAIC_PROB:
        return _mosaic(imgs, labels)
    if r < MOSAIC_PROB + MIXUP_PROB:
        return _mixup(imgs, labels)
    return imgs, labels

# ── pos_weight (desbalance multietiqueta) ──
def compute_pos_weight(samples, n_classes, cap=POS_WEIGHT_MAX):
    labels = np.stack([l for _, l in samples])
    pos = labels.sum(0).clip(min=1)
    w = np.clip((len(labels) - pos) / pos, 1.0, cap)
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)

pos_weight = compute_pos_weight(train_samples, NUM_CLASSES)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print("pos_weight:", {c: round(w, 1) for c, w in zip(CLASSES, pos_weight.tolist())})

_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

# ── Métricas ──
def compute_metrics(y_true, y_logits, thr=None):
    thr = THRESHOLD if thr is None else np.asarray(thr)
    probs = expit(y_logits)
    y_pred = (probs >= thr).astype(int)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    aps = [average_precision_score(y_true[:, c], probs[:, c])
           for c in range(y_true.shape[1]) if y_true[:, c].sum() > 0]
    return f1, (np.mean(aps) if aps else 0.0)

def tune_thresholds(y_true, y_logits):
    probs, out = expit(y_logits), []
    for c in range(y_true.shape[1]):
        bt, bf = THRESHOLD, -1.0
        for t in np.arange(0.05, 0.96, 0.01):
            f = f1_score(y_true[:, c], (probs[:, c] >= t).astype(int), zero_division=0)
            if f > bf:
                bf, bt = f, t
        out.append(round(float(bt), 2))
    return out

# ── Train / Eval con acumulación de gradiente ──
def train_epoch(model, loader, optimizer, accum=1):
    model.train()
    total, eps, i = 0.0, LABEL_SMOOTH, -1
    optimizer.zero_grad(set_to_none=True)
    for i, (imgs, labels) in enumerate(loader):
        imgs   = imgs.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.no_grad():
            imgs = gpu_aug_train(imgs)                    # aug por imagen (en [0,1])
            imgs, labels = mix_crossover(imgs, labels)    # mosaico 2x2 / MixUp -> etiqueta union (crossover)
            imgs = normalize_gpu(imgs).to(memory_format=torch.channels_last)
        labels = labels * (1 - eps) + 0.5 * eps          # label smoothing
        with torch.autocast('cuda', dtype=_DTYPE):
            loss = criterion(model(imgs), labels) / accum
        loss.backward()                                  # BF16 no necesita GradScaler
        if (i + 1) % accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)   # v3: estabiliza el step
            optimizer.step(); optimizer.zero_grad(set_to_none=True)
        total += loss.item() * accum
        if (i + 1) % 100 == 0:
            print(f"  batch {i+1}/{len(loader)}", end='\r')
    if (i + 1) % accum != 0:                             # flush de gradientes sobrantes
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); optimizer.zero_grad(set_to_none=True)
    return total / max(1, len(loader))

@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    total, yt, yl = 0.0, [], []
    for imgs, labels in loader:
        imgs   = imgs.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', dtype=_DTYPE):
            logits = model(imgs)
            total += criterion(logits, labels).item()
        yt.append(labels.cpu().numpy()); yl.append(logits.float().cpu().numpy())
    if not yt:
        return 0.0, 0.0, 0.0, None, None
    y_true, y_logits = np.concatenate(yt), np.concatenate(yl)
    f1, mAP = compute_metrics(y_true, y_logits)
    return total / len(loader), f1, mAP, y_true, y_logits

# ── Checkpoints (escritura atómica: .tmp + rename, por si se corta a media escritura) ──
def _state(model):
    return {k: v.detach().cpu().clone() for k, v in _orig(model).state_dict().items()}

def _atomic_save(payload, path):
    tmp = path + '.tmp'
    torch.save(payload, tmp)
    os.replace(tmp, path)

def save_best(model, epoch, best_f1, phase, history, thresholds=None):
    _atomic_save({
        'model_state_dict': _state(model), 'model_name': MODEL_ARCH,
        'classes': CLASSES, 'num_classes': NUM_CLASSES,
        'best_f1': best_f1, 'epoch': epoch, 'phase': phase, 'history': history,
        'threshold': THRESHOLD, 'thresholds': thresholds,
        'img_size': IMG_SIZE, 'mean': IMAGENET_MEAN, 'std': IMAGENET_STD,
    }, SAVE_PATH)

def save_resume(model, epoch, phase, best_f1, since_best, history, thresholds=None):
    _atomic_save({
        'model_state_dict': _state(model), 'model_name': MODEL_ARCH,
        'classes': CLASSES, 'num_classes': NUM_CLASSES,
        'epoch': epoch, 'phase': phase, 'best_f1': best_f1,
        'since_best': since_best, 'history': history, 'thresholds': thresholds,
    }, CKPT_RESUME)

print(f"Funciones listas | dtype={_DTYPE} | accum={GRAD_ACCUM} | clip={GRAD_CLIP} | crossover: mosaico+mixup")

## Celda 9 — ENTRENAMIENTO (una sola fase, a prueba de cortes)

> **v3:** cada batch se arma como **mosaico 2×2** (4 series), **MixUp** (2 series) o **limpio** (1 serie) — ver `mix_crossover` en la Celda 8. Así el modelo ve crossovers sintéticos con etiqueta unión y aprende a encender varias clases a la vez, sin necesitar un dataset de crossovers reales.
>
> **v2:** una sola fase de fine-tuning (sin el cambio de `requires_grad` a media corrida que rompia v1; el `compile` ya quedo trazado con la config final en la Celda 7).
>
> **Si Colab se desconecta:** vuelve a ejecutar **solo esta celda** -> reanuda desde la ultima epoca. Para empezar de cero, borra `toonverse_resume_v3.pt` de tu carpeta `models`.

In [ ]:
# ════════════════════════════════════════════════════════════════
# ENTRENAMIENTO una sola fase — guarda estado en Drive CADA epoca.
# (el compile ya se hizo en la Celda 7 con la config final de entrenables)
# ════════════════════════════════════════════════════════════════
optimizer = AdamW(trainable_params(model), lr=LR_FT, weight_decay=WD_FT)
warm = LinearLR(optimizer, start_factor=0.1, total_iters=1)          # 1 epoca de warmup
cos  = CosineAnnealingLR(optimizer, T_max=max(1, NUM_EPOCHS - 1))
scheduler = SequentialLR(optimizer, [warm, cos], milestones=[1])

# ── Estado inicial / reanudacion ──
history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_mAP': []}
best_f1, since_best, start_epoch = 0.0, 0, 1

if os.path.exists(CKPT_RESUME):
    ck = torch.load(CKPT_RESUME, map_location='cpu', weights_only=False)
    _orig(model).load_state_dict(ck['model_state_dict'])
    start_epoch = ck['epoch'] + 1
    best_f1, since_best, history = ck['best_f1'], ck['since_best'], ck['history']
    for _ in range(start_epoch - 1):          # alinear scheduler al reanudar
        scheduler.step()
    print(f"REANUDANDO desde epoca {start_epoch}/{NUM_EPOCHS} (mejor F1 = {best_f1:.4f})")
else:
    print(f"Inicio limpio — {NUM_EPOCHS} epocas (afina desde stage {UNFREEZE_FROM_STAGE})")

best_state = _state(model)
t_run = time.time()

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    t0 = time.time()
    tr_loss = train_epoch(model, train_loader, optimizer, accum=GRAD_ACCUM)
    vl_loss, vl_f1, vl_mAP, _, _ = eval_epoch(model, val_loader)
    scheduler.step()
    dt = (time.time() - t0) / 60
    for k, v in zip(['train_loss', 'val_loss', 'val_f1', 'val_mAP'],
                    [tr_loss, vl_loss, vl_f1, vl_mAP]):
        history[k].append(v)

    improved = vl_f1 > best_f1
    if improved:
        best_f1, since_best = vl_f1, 0
        best_state = _state(model)
        save_best(model, epoch, best_f1, 'ft', history)
    else:
        since_best += 1

    save_resume(model, epoch, 'ft', best_f1, since_best, history)   # SIEMPRE

    lr_now = optimizer.param_groups[0]['lr']
    print(f"[E{epoch:02d}/{NUM_EPOCHS}] {dt:.1f}min | "
          f"loss {tr_loss:.4f}/{vl_loss:.4f} | F1 {vl_f1:.4f} | mAP {vl_mAP:.4f} "
          f"| lr {lr_now:.2e}" + ("  <-- mejor" if improved else ""))

    if since_best >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stop ({EARLY_STOP_PATIENCE} epocas sin mejora)")
        break

print(f"\nMejor F1: {best_f1:.4f} | tiempo total {(time.time()-t_run)/3600:.1f} h")
print(f"Modelo -> {SAVE_PATH}")

Inicio limpio — 14 epocas (afina desde stage 3)
[E01/14] 22.2min | loss 0.5861/0.2379 | F1 0.7767 | mAP 0.9264 | lr 1.20e-04  <-- mejor
[E02/14] 20.1min | loss 0.4774/0.1718 | F1 0.9137 | mAP 0.9813 | lr 1.18e-04  <-- mejor
[E03/14] 20.0min | loss 0.4499/0.1626 | F1 0.9362 | mAP 0.9868 | lr 1.13e-04  <-- mejor
[E04/14] 20.1min | loss 0.4398/0.1582 | F1 0.9452 | mAP 0.9890 | lr 1.05e-04  <-- mejor
[E05/14] 20.1min | loss 0.4340/0.1568 | F1 0.9490 | mAP 0.9893 | lr 9.41e-05  <-- mejor
[E06/14] 20.2min | loss 0.4299/0.1537 | F1 0.9572 | mAP 0.9903 | lr 8.13e-05  <-- mejor
[E07/14] 20.2min | loss 0.4272/0.1532 | F1 0.9586 | mAP 0.9901 | lr 6.72e-05  <-- mejor
[E08/14] 20.1min | loss 0.4251/0.1543 | F1 0.9601 | mAP 0.9892 | lr 5.28e-05  <-- mejor
[E09/14] 20.1min | loss 0.4236/0.1527 | F1 0.9629 | mAP 0.9897 | lr 3.87e-05  <-- mejor
[E10/14] 20.2min | loss 0.4225/0.1523 | F1 0.9651 | mAP 0.9903 | lr 2.59e-05  <-- mejor


## Celda 10 — Resultados, umbrales por clase y curvas

In [ ]:
# Evaluacion final con los mejores pesos + umbrales por clase
_orig(model).load_state_dict(best_state)
_, _, _, y_true, y_logits = eval_epoch(model, val_loader)
thresholds = tune_thresholds(y_true, y_logits)
f1_tuned, mAP_f = compute_metrics(y_true, y_logits, thresholds)
final_f1 = max(best_f1, f1_tuned)

print("Umbrales por clase:")
for c, t in zip(CLASSES, thresholds):
    print(f"  {c:22s}: {t:.2f}")
print(f"\n  F1 @0.5      : {best_f1:.4f}")
print(f"  F1 ajustado  : {f1_tuned:.4f}  ({f1_tuned-best_f1:+.4f})")

# Guarda el modelo final con umbrales (para la app)
save_best(model, len(history['val_f1']), final_f1, 'final', history, thresholds)

# Curvas
ep = range(1, len(history['train_loss']) + 1)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f"ToonVerse ConvNeXt V2-L (v3) | {NUM_CLASSES} cls | "
             f"{len(all_samples):,} imgs | F1 {final_f1:.4f} mAP {mAP_f:.4f}")
ax[0].plot(ep, history['train_loss'], 'b-o', ms=3, label='train')
ax[0].plot(ep, history['val_loss'],   'r-o', ms=3, label='val')
ax[0].set_title('Loss'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep, history['val_f1'], 'g-o', ms=3)
ax[1].axhline(0.85, color='green', ls='--', label='meta 0.85')
ax[1].set_title('Macro F1'); ax[1].set_ylim(0, 1); ax[1].legend(); ax[1].grid(alpha=.3)
ax[2].plot(ep, history['val_mAP'], 'm-o', ms=3)
ax[2].set_title('mAP'); ax[2].set_ylim(0, 1); ax[2].grid(alpha=.3)
for a in ax:
    a.set_xlabel('epoca')
plt.tight_layout()
plt.savefig(SAVE_PATH.replace('.pt', '_curvas.png'), dpi=100, bbox_inches='tight')
plt.show()

print(f"\n{'='*65}\nRESUMEN\n{'='*65}")
print(f"Clases: {NUM_CLASSES} | Imgs: {len(all_samples):,} | F1: {final_f1:.4f} | mAP: {mAP_f:.4f}")

## Celda 10b — Sanity-check de crossovers (sintético)

La validación es **single-label**, así que su F1/mAP **no** mide si el modelo detecta crossovers. Esta celda arma mosaicos 2×2 con imágenes reales de val (varios universos en una sola imagen) y verifica que el modelo encienda **≥2 clases correctas** a la vez — justo el caso del póster Simpson + Star Wars.

In [ ]:
# ── Sanity-check: ¿detecta crossovers? Arma mosaicos 2x2 con imágenes reales de val ──
_orig(model).load_state_dict(best_state)
model.eval()
_thr = np.asarray(thresholds)

imgs, labels = next(iter(val_loader))                       # val viene NORMALIZADO
imgs   = imgs.to(DEVICE, memory_format=torch.channels_last)
labels = labels.to(DEVICE)
den    = (imgs * _std + _mean).clamp(0, 1)                  # des-normaliza -> [0,1] para mosaico
mos, lab = _mosaic(den, labels)                             # 4 universos por imagen + etiqueta union

with torch.no_grad(), torch.autocast('cuda', dtype=_DTYPE):
    probs = torch.sigmoid(_orig(model)(normalize_gpu(mos))).float().cpu().numpy()
lab = lab.cpu().numpy()

print("="*65); print("SANITY-CHECK CROSSOVERS (mosaicos 2x2 de val)"); print("="*65)
ok = 0
N  = min(10, len(mos))
for i in range(N):
    real = [CLASSES[j] for j in range(NUM_CLASSES) if lab[i, j] > 0.5]
    pred = [CLASSES[j] for j in range(NUM_CLASSES) if probs[i, j] >= _thr[j]]
    aciertos = set(real) & set(pred)
    multi = len(aciertos) >= 2                              # >=2 universos correctos = crossover OK
    ok += multi
    print(f"  {'OK ' if multi else '.. '} real={real}")
    print(f"      detectado={pred}")
print(f"\nMosaicos con >=2 universos correctos: {ok}/{N}")
print("(si la mayoría son OK, el modelo ya co-detecta crossovers como en el póster Simpson+Star Wars)")

## Celda 11 — Inferencia en una imagen (para tu app)

> **Importante:** tu app cargaba **MobileNetV2**. Ahora el checkpoint es **ConvNeXt V2-Large**, así que la app debe construir el modelo con `timm` igual que aquí (el nombre de arquitectura va guardado dentro del `.pt`).

In [ ]:
from PIL import Image as PILImage

def load_model_for_inference(ckpt_path, device='cuda'):
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    m = timm.create_model(ck['model_name'], pretrained=False, num_classes=ck['num_classes'])
    m.load_state_dict(ck['model_state_dict'])
    m.to(device).eval()
    tf = transforms.Compose([
        transforms.Resize((ck['img_size'], ck['img_size'])),
        transforms.ToTensor(),
        transforms.Normalize(ck['mean'], ck['std']),
    ])
    thr = np.asarray(ck.get('thresholds') or [ck['threshold']] * ck['num_classes'])
    return m, tf, ck['classes'], thr

@torch.no_grad()
def predict_image(path, m, tf, classes, thr, device='cuda'):
    x = tf(PILImage.open(path).convert('RGB')).unsqueeze(0).to(device)
    probs = torch.sigmoid(m(x)).cpu().numpy()[0]
    hits = [(classes[i], float(probs[i])) for i in range(len(classes)) if probs[i] >= thr[i]]
    hits.sort(key=lambda h: -h[1])
    return hits, probs

# Ejemplo de uso:
# m, tf, classes, thr = load_model_for_inference(SAVE_PATH)
# hits, probs = predict_image('/content/test.jpg', m, tf, classes, thr)
# print("Universos detectados:", hits)   # p.ej. [('naruto',0.97),('bleach',0.62)] = crossover